# Zomato Dataset Analysis
End-to-End Data Science Project for Alfido Tech
By the Intern.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic styling
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Data Loading
*Make sure to download `zomato.csv` from Kaggle and place it in the same directory.*

In [ ]:
# Load dataset
file_path = 'zomato.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded with shape {df.shape}")
except Exception as e:
    print("Dataset not found. Please download from kaggle.com/datasets/bhanupratapbiswas/zomato")
    
# Display top 5 rows if available
if 'df' in locals():
    display(df.head())


## 2. Basic EDA & Data Cleaning
Handling missing values, text parsing, and type conversion.

In [ ]:
if 'df' in locals():
    # Drop irrelevant columns
    columns_to_drop = ['url', 'address', 'phone', 'dish_liked', 'reviews_list', 'menu_item']
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], axis=1)

    # Rename columns for convenience
    df.rename(columns={'approx_cost(for two people)':'cost', 'listed_in(type)':'type', 'listed_in(city)':'city'}, inplace=True)
    
    # Clean the rating column: '4.1/5' -> 4.1
    def clean_rate(value):
        if pd.isnull(value):
            return np.nan
        value = str(value)
        if value == '- ' or value == 'NEW' or value == '-':
            return np.nan
        value = value.split('/')[0].strip()
        try:
            return float(value)
        except:
            return np.nan

    df['rate'] = df['rate'].apply(clean_rate)
    
    # Clean cost column: '1,200' -> 1200
    def clean_cost(value):
        if pd.isnull(value):
            return np.nan
        value = str(value)
        value = value.replace(',', '')
        try:
            return float(value)
        except:
            return np.nan

    df['cost'] = df['cost'].apply(clean_cost)
    
    # Drop NAs
    df.dropna(inplace=True)
    print("Data cleaning complete. Remaining records:", df.shape[0])


## 3. Exploratory Data Analysis & Visualizations
Cuisine vs rating, Location Hotspots, Price vs Rating, Heatmaps, Wordclouds.

In [ ]:
if 'df' in locals():
    # 1. Location Hotspots
    plt.figure(figsize=(12, 8))
    location_counts = df['location'].value_counts()[:15]
    sns.barplot(x=location_counts, y=location_counts.index, palette='viridis')
    plt.title('Top 15 Neighborhoods with the Most Restaurants')
    plt.xlabel('Number of Restaurants')
    plt.ylabel('Location')
    plt.show()

    # 2. Price vs Rating
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='cost', y='rate', data=df, alpha=0.5, color='orange')
    plt.title('Cost for Two vs. Rating')
    plt.xlabel('Cost for Two (INR)')
    plt.ylabel('Rating')
    plt.show()

    # 3. Heatmap of Correlations
    plt.figure(figsize=(8,6))
    numeric_df = df[['rate', 'votes', 'cost']].dropna()
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Heatmap')
    plt.show()

    # 4. Wordcloud for Popular Cuisines
    plt.figure(figsize=(10,10))
    cuisines_text = " ".join(df['cuisines'].astype(str))
    wordcloud = WordCloud(width=800, height=800, background_color='white', min_font_size=10).generate(cuisines_text)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title("Most Common Cuisines WordCloud")
    plt.show()


## 4. Machine Learning Model
Predicting a restaurant's rating based on location, cost, and votes.

In [ ]:
if 'df' in locals():
    # Select features for mapping
    ml_df = df[['location', 'rest_type', 'cost', 'votes', 'rate']].dropna().copy()
    
    # Label Encoding categorical variables
    le_loc = LabelEncoder()
    le_type = LabelEncoder()
    
    ml_df['location'] = le_loc.fit_transform(ml_df['location'])
    ml_df['rest_type'] = le_type.fit_transform(ml_df['rest_type'])
    
    X = ml_df.drop('rate', axis=1)
    y = ml_df['rate']
    
    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Build Model
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    
    # Predictions
    preds = rf_model.predict(X_test)
    
    print("Model Evaluation:")
    print("MSE:", mean_squared_error(y_test, preds))
    print("R2 Score:", r2_score(y_test, preds))
    
    # Feature Importance
    importances = rf_model.feature_importances_
    feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)
    plt.figure(figsize=(8,4))
    sns.barplot(x='Importance', y='Feature', data=feat_df, palette='magma')
    plt.title('Feature Importance in Predicting Ratings')
    plt.show()


## 5. Conclusions & Recommendations
*(Reflected fully in the PDF Report)*
1. Partner with higher-rated cuisines (e.g. Continental, North Indian).
2. Expand delivery density in hotspots like BTM and Koramangala.
3. Content ideas: 'Hidden Gems under Rs 500' based on cost vs rating spread.
4. Focus marketing on restaurants allowing table booking/online orders.
5. Encourage users to leave votes, as high engagement correlates strongly with reliable ratings!